# CNN Spatio-Temporal Stream — Deepfake Detection

## Two-Stream Late Fusion Architecture (Stream 2 of 2)

This notebook implements a **research-grade Spatio-Temporal CNN** using EfficientNet-B4 backbone with **BiLSTM Temporal Aggregation** for deepfake detection.

### Key Research Contributions:

1. **Temporal Modeling (BiLSTM + Multi-Head Attention)**
   - Unlike naive frame averaging, we model inter-frame dependencies
   - Detects temporal flickering, blending shifts, and motion anomalies
   - Enables detection of GAN/Diffusion artifacts that manifest across frames

2. **Grad-CAM Visualization**
   - Provides visual proof of what the model learns
   - Shows attention on facial regions (jawline, eyes, blending boundaries)
   - Essential for research paper methodology section

```
┌──────────────────────────────────────────────────────────────────────────────┐
│                    SPATIO-TEMPORAL ARCHITECTURE                              │
├──────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│   Video (T frames) ──→ MTCNN Face Detection ──→ T × (224, 224, 3) crops     │
│                                                                              │
│   ┌─────────────────────────────────────────────────────────────────────┐   │
│   │  SPATIAL FEATURE EXTRACTION                                          │   │
│   │  EfficientNet-B4 (pretrained, 1792-dim features per frame)          │   │
│   │                                                                      │   │
│   │      Frame 1 ──→ [f₁]                                               │   │
│   │      Frame 2 ──→ [f₂]     ──→ Feature Sequence (T × 1792)           │   │
│   │      ...                                                             │   │
│   │      Frame T ──→ [fₜ]                                               │   │
│   └─────────────────────────────────────────────────────────────────────┘   │
│                              ↓                                               │
│   ┌─────────────────────────────────────────────────────────────────────┐   │
│   │  TEMPORAL AGGREGATION (BiLSTM + Multi-Head Attention)               │   │
│   │                                                                      │   │
│   │  [f₁, f₂, ..., fₜ] ──→ BiLSTM (2-layer, bidirectional)              │   │
│   │                                ↓                                     │   │
│   │                    Multi-Head Self-Attention (4 heads)               │   │
│   │                                ↓                                     │   │
│   │                    Weighted Temporal Pooling                         │   │
│   │                                ↓                                     │   │
│   │                    Video-Level Representation (512-dim)              │   │
│   └─────────────────────────────────────────────────────────────────────┘   │
│                              ↓                                               │
│   ┌─────────────────────────────────────────────────────────────────────┐   │
│   │  CLASSIFIER                                                          │   │
│   │  Linear(512 → 256) → BatchNorm → GELU → Dropout                     │   │
│   │  Linear(256 → 128) → BatchNorm → GELU → Dropout                     │   │
│   │  Linear(128 → 1) → Sigmoid                                          │   │
│   │                        ↓                                             │   │
│   │                    P_CNN (0 = Real, 1 = Fake)                        │   │
│   └─────────────────────────────────────────────────────────────────────┘   │
│                                                                              │
└──────────────────────────────────────────────────────────────────────────────┘

                    ↓ LATE FUSION ↓

        P_final = w₁ × P_CNN + w₂ × P_rPPG (from Stream 1)
```

**Output:** `cnn_predictions.csv` with video-level P_CNN scores for Late Fusion

## 1. Setup & Imports

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# INSTALL DEPENDENCIES
# ═══════════════════════════════════════════════════════════════════════════════

import subprocess
import sys

def install_packages():
    packages = [
        "facenet-pytorch",
        "timm",
        "albumentations",
        "opencv-python-headless",
    ]
    for pkg in packages:
        print(f"Installing {pkg}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    print("✓ All packages installed")

install_packages()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════

import os
import gc
import cv2
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from typing import List, Tuple, Dict, Optional
from tqdm.auto import tqdm
import io
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
# PyTorch 2.0+ compatible AMP imports
try:
    from torch.amp import autocast, GradScaler
    USE_NEW_AMP = True
except ImportError:
    from torch.cuda.amp import autocast, GradScaler
    USE_NEW_AMP = False

import timm
from facenet_pytorch import MTCNN
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report

warnings.filterwarnings('ignore')

# ─── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ─── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION (Research Paper Ready)
# ═══════════════════════════════════════════════════════════════════════════════

import math

class Config:
    # Experiment tracking (for reproducibility)
    EXPERIMENT_NAME = "CNN_SpatioTemporal_BiLSTM_Attn"
    EXPERIMENT_VERSION = "v2.0_research"
    
    # Dataset paths
    DATASET_ROOT = "/kaggle/input/400videoseach/content/drive/MyDrive/face_dataset_dip"
    REAL_DIR = os.path.join(DATASET_ROOT, "real_videos")
    FAKE_DIR = os.path.join(DATASET_ROOT, "deepfake_videos")
    OUTPUT_DIR = "/kaggle/working"

    # Frame extraction
    FRAMES_PER_VIDEO = 15
    IMG_SIZE = 224

    # Training (P100 optimized)
    BATCH_SIZE = 8  # Video-level batching (8 videos × 15 frames = 120 images/batch)
    NUM_EPOCHS = 20
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-2
    NUM_WORKERS = 2
    WARMUP_RATIO = 0.1  # 10% warmup steps
    LABEL_SMOOTHING = 0.1  # For better calibration

    # Model - Backbone
    MODEL_NAME = "efficientnet_b4"
    DROPOUT = 0.4
    HIDDEN_DIM = 256
    USE_GRADIENT_CHECKPOINTING = True  # Saves ~40% VRAM on P100

    # Model - Temporal Aggregator (BiLSTM for inter-frame temporal modeling)
    TEMPORAL_TYPE = "bilstm_attention"  # Options: "bilstm", "bilstm_attention", "transformer"
    LSTM_HIDDEN = 256
    LSTM_LAYERS = 2
    ATTENTION_HEADS = 4
    FREEZE_BACKBONE = False  # Set True for faster training, False for best accuracy

    # K-Fold Cross-Validation (RESEARCH REQUIREMENT)
    K_FOLDS = 5
    CURRENT_FOLD = 0  # Which fold to run (0-4), or set to -1 for single split
    
    # Split (used when CURRENT_FOLD = -1)
    TRAIN_RATIO = 0.8
    VAL_RATIO = 0.2
    
    @classmethod
    def to_dict(cls):
        """Export config for logging and reproducibility."""
        return {k: v for k, v in vars(cls).items() 
                if not k.startswith('_') and not callable(v)}

cfg = Config()

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# Save config for reproducibility
import json
config_path = os.path.join(cfg.OUTPUT_DIR, 'config.json')
with open(config_path, 'w') as f:
    json.dump(cfg.to_dict(), f, indent=2, default=str)

print("="*70)
print(f"CNN SPATIO-TEMPORAL STREAM: {cfg.EXPERIMENT_NAME} {cfg.EXPERIMENT_VERSION}")
print("="*70)
print(f"  Dataset root: {cfg.DATASET_ROOT}")
print(f"  Frames per video: {cfg.FRAMES_PER_VIDEO}")
print(f"  Image size: {cfg.IMG_SIZE}x{cfg.IMG_SIZE}")
print(f"  Batch size: {cfg.BATCH_SIZE} (video-level)")
print(f"  Epochs: {cfg.NUM_EPOCHS}")
print(f"  Learning rate: {cfg.LEARNING_RATE} (with {cfg.WARMUP_RATIO*100:.0f}% warmup)")
print(f"  Label smoothing: {cfg.LABEL_SMOOTHING}")
print(f"  Backbone: {cfg.MODEL_NAME}")
print(f"  Gradient checkpointing: {cfg.USE_GRADIENT_CHECKPOINTING}")
print(f"  Temporal aggregator: {cfg.TEMPORAL_TYPE}")
print(f"  LSTM hidden: {cfg.LSTM_HIDDEN}, layers: {cfg.LSTM_LAYERS}")
print(f"  K-Fold CV: {cfg.K_FOLDS} folds (current fold: {cfg.CURRENT_FOLD})")
print(f"  Config saved to: {config_path}")
print("="*70)


## 2. Frame Extraction & Face Detection

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MTCNN FACE DETECTOR
# ═══════════════════════════════════════════════════════════════════════════════

class FaceExtractor:
    """
    Extracts faces from video frames using MTCNN.
    Falls back to center crop if no face is detected.
    """
    
    def __init__(self, device, img_size=224, margin=40):
        self.device = device
        self.img_size = img_size
        self.margin = margin
        
        # Initialize MTCNN with optimized settings
        self.mtcnn = MTCNN(
            image_size=img_size,
            margin=margin,
            min_face_size=60,
            thresholds=[0.6, 0.7, 0.7],
            factor=0.709,
            post_process=False,  # Raw pixel values [0, 255], no [-1, 1] normalization
            device=device,
            keep_all=False,  # Only largest face
        )
        print(f"✓ MTCNN initialized on {device}")
    
    def extract_face(self, frame: np.ndarray) -> Optional[np.ndarray]:
        """
        Extract face from a single frame.
        Returns: Face crop as numpy array (H, W, 3) or None if failed.
        """
        try:
            # Convert BGR to RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(frame_rgb)
            
            # Detect face
            face = self.mtcnn(pil_img)
            
            if face is not None:
                # MTCNN returns tensor (C, H, W), convert to numpy (H, W, C)
                # Since post_process=False, values are already 0-255
                face_np = face.permute(1, 2, 0).cpu().numpy().astype(np.uint8)
                return face_np
            else:
                # Fallback: center crop
                return self._center_crop(frame_rgb)
                
        except Exception as e:
            # Fallback: center crop
            return self._center_crop(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    
    def _center_crop(self, frame: np.ndarray) -> np.ndarray:
        """Center crop fallback when face detection fails."""
        h, w = frame.shape[:2]
        size = min(h, w)
        y = (h - size) // 2
        x = (w - size) // 2
        crop = frame[y:y+size, x:x+size]
        return cv2.resize(crop, (self.img_size, self.img_size))


# Initialize face extractor
face_extractor = FaceExtractor(DEVICE, img_size=cfg.IMG_SIZE)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VIDEO FRAME EXTRACTION
# ═══════════════════════════════════════════════════════════════════════════════

def extract_frames_from_video(video_path: str, n_frames: int = 10) -> List[np.ndarray]:
    """
    Extract n evenly spaced frames from a video.
    
    Args:
        video_path: Path to video file
        n_frames: Number of frames to extract
        
    Returns:
        List of frame arrays (BGR format)
    """
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        return []
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        return []  # Corrupted video
    
    if total_frames < n_frames:
        # If video has fewer frames, extract all
        frame_indices = list(range(total_frames))
    else:
        # Evenly spaced frame indices
        frame_indices = np.linspace(0, total_frames - 1, n_frames, dtype=int)
    
    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(frame)
    
    cap.release()
    return frames


def process_video(video_path: str, face_extractor: FaceExtractor, n_frames: int = 10) -> List[np.ndarray]:
    """
    Extract frames and detect faces from a video.
    
    Returns:
        List of face crops as numpy arrays
    """
    frames = extract_frames_from_video(video_path, n_frames)
    
    face_crops = []
    for frame in frames:
        face = face_extractor.extract_face(frame)
        if face is not None:
            face_crops.append(face)
    
    return face_crops


print("✓ Frame extraction functions defined")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PROCESS ALL VIDEOS
# ═══════════════════════════════════════════════════════════════════════════════

def collect_video_paths():
    """Collect all video paths with labels."""
    video_extensions = ('.mp4', '.avi', '.mov', '.mkv')
    
    videos = []
    
    # Real videos (label = 0)
    if os.path.exists(cfg.REAL_DIR):
        for f in sorted(os.listdir(cfg.REAL_DIR)):
            if f.lower().endswith(video_extensions):
                videos.append({
                    'video_id': f,
                    'path': os.path.join(cfg.REAL_DIR, f),
                    'label': 0  # Real
                })
    
    # Fake videos (label = 1)
    if os.path.exists(cfg.FAKE_DIR):
        for f in sorted(os.listdir(cfg.FAKE_DIR)):
            if f.lower().endswith(video_extensions):
                videos.append({
                    'video_id': f,
                    'path': os.path.join(cfg.FAKE_DIR, f),
                    'label': 1  # Fake
                })
    
    return videos

# Collect videos
all_videos = collect_video_paths()
print(f"Total videos found: {len(all_videos)}")
print(f"  Real: {sum(1 for v in all_videos if v['label'] == 0)}")
print(f"  Fake: {sum(1 for v in all_videos if v['label'] == 1)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXTRACT ALL FACES (with caching)
# ═══════════════════════════════════════════════════════════════════════════════

def extract_all_faces(videos: List[dict], face_extractor: FaceExtractor) -> Dict[str, List[np.ndarray]]:
    """
    Extract faces from all videos.
    
    Returns:
        Dictionary mapping video_id to list of face crops
    """
    face_data = {}
    
    print("\n" + "="*70)
    print("EXTRACTING FACES FROM VIDEOS")
    print("="*70)
    
    failed_videos = []
    
    for video in tqdm(videos, desc="Processing videos"):
        video_id = video['video_id']
        video_path = video['path']
        
        faces = process_video(video_path, face_extractor, cfg.FRAMES_PER_VIDEO)
        
        if len(faces) >= 3:  # Require at least 3 valid faces
            face_data[video_id] = faces
        else:
            failed_videos.append(video_id)
    
    print(f"\n✓ Successfully processed: {len(face_data)} videos")
    print(f"✗ Failed/insufficient faces: {len(failed_videos)} videos")
    
    # Memory cleanup
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return face_data

# Extract faces
face_data = extract_all_faces(all_videos, face_extractor)

# DO NOT shrink all_videos — this would misalign train_test_split indices
# with the ML notebook. Downstream code (Dataset, predict) already checks
# `if video_id in face_data` to safely skip missing videos.

# Validate we have face data
if len(face_data) == 0:
    raise ValueError("ERROR: No videos with valid face detections! Check face extraction.")
print(f"\nVideos with valid faces: {len(face_data)}/{len(all_videos)}")

## 3. Dataset Creation & Data Leakage Prevention

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# K-FOLD CROSS-VALIDATION SETUP (RESEARCH REQUIREMENT)
# ═══════════════════════════════════════════════════════════════════════════════
#
# CRITICAL FOR RESEARCH: Single train/val split is insufficient for publication.
# Papers require K-fold CV with confidence intervals to demonstrate robustness.
#
# This cell supports two modes:
#   1. K-Fold mode (CURRENT_FOLD = 0-4): Trains on specified fold
#   2. Single split mode (CURRENT_FOLD = -1): Simple train/val split
#
# For full K-Fold CV, run notebook 5 times with CURRENT_FOLD = 0, 1, 2, 3, 4
# ═══════════════════════════════════════════════════════════════════════════════

from sklearn.model_selection import StratifiedKFold

def create_kfold_splits(videos: List[dict], n_splits: int = 5, seed: int = 42):
    """
    Create K-Fold stratified splits for rigorous evaluation.
    
    CRITICAL: Ensures no video appears in both train and val within any fold.
    Uses stratification to maintain class balance across folds.
    
    Args:
        videos: List of video dicts with 'video_id' and 'label'
        n_splits: Number of folds (typically 5 or 10)
        seed: Random seed for reproducibility
    
    Returns:
        List of fold dicts, each with 'train' and 'val' video lists
    """
    labels = np.array([v['label'] for v in videos])
    indices = np.arange(len(videos))
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    folds = []
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(indices, labels)):
        train_videos = [videos[i] for i in train_idx]
        val_videos = [videos[i] for i in val_idx]
        
        # Verify no data leakage
        train_ids = set(v['video_id'] for v in train_videos)
        val_ids = set(v['video_id'] for v in val_videos)
        assert len(train_ids & val_ids) == 0, f"DATA LEAKAGE in fold {fold_idx}!"
        
        folds.append({
            'fold': fold_idx,
            'train': train_videos,
            'val': val_videos,
            'train_real': sum(1 for v in train_videos if v['label'] == 0),
            'train_fake': sum(1 for v in train_videos if v['label'] == 1),
            'val_real': sum(1 for v in val_videos if v['label'] == 0),
            'val_fake': sum(1 for v in val_videos if v['label'] == 1),
        })
    
    return folds


def split_videos_single(videos: List[dict], train_ratio: float = 0.8, seed: int = 42):
    """
    Simple stratified train/val split (for quick experiments).
    Use K-Fold for final publication results.
    """
    labels = [v['label'] for v in videos]
    train_videos, val_videos = train_test_split(
        videos, train_size=train_ratio, random_state=seed, stratify=labels
    )
    return train_videos, val_videos


# ═══════════════════════════════════════════════════════════════════════════════
# CREATE SPLITS
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("DATA SPLIT CONFIGURATION")
print("="*70)

if cfg.CURRENT_FOLD >= 0:
    # K-Fold mode
    print(f"\n📊 K-FOLD CROSS-VALIDATION MODE (Fold {cfg.CURRENT_FOLD + 1}/{cfg.K_FOLDS})")
    print("-"*50)
    
    kfold_splits = create_kfold_splits(all_videos, n_splits=cfg.K_FOLDS, seed=SEED)
    
    # Print all fold statistics
    print("\nFold Statistics:")
    for fold in kfold_splits:
        print(f"  Fold {fold['fold'] + 1}: Train={len(fold['train'])} (R:{fold['train_real']}, F:{fold['train_fake']}), "
              f"Val={len(fold['val'])} (R:{fold['val_real']}, F:{fold['val_fake']})")
    
    # Select current fold
    current_split = kfold_splits[cfg.CURRENT_FOLD]
    train_videos = current_split['train']
    val_videos = current_split['val']
    
    print(f"\n✓ Using Fold {cfg.CURRENT_FOLD + 1}")
    print(f"  Train: {len(train_videos)} videos (Real: {current_split['train_real']}, Fake: {current_split['train_fake']})")
    print(f"  Val: {len(val_videos)} videos (Real: {current_split['val_real']}, Fake: {current_split['val_fake']})")
    
else:
    # Single split mode
    print(f"\n⚡ SINGLE SPLIT MODE (Quick Experiment)")
    print("-"*50)
    print("⚠ Note: Use K-Fold mode (CURRENT_FOLD >= 0) for publication results")
    
    train_videos, val_videos = split_videos_single(all_videos, cfg.TRAIN_RATIO, SEED)
    
    print(f"\n  Train: {len(train_videos)} videos")
    print(f"    Real: {sum(1 for v in train_videos if v['label'] == 0)}")
    print(f"    Fake: {sum(1 for v in train_videos if v['label'] == 1)}")
    print(f"\n  Val: {len(val_videos)} videos")
    print(f"    Real: {sum(1 for v in val_videos if v['label'] == 0)}")
    print(f"    Fake: {sum(1 for v in val_videos if v['label'] == 1)}")

# Verify no leakage
train_ids = set(v['video_id'] for v in train_videos)
val_ids = set(v['video_id'] for v in val_videos)
assert len(train_ids & val_ids) == 0, "DATA LEAKAGE DETECTED!"
print("\n✅ NO DATA LEAKAGE: Train and Val sets are completely separate")
print("="*70)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATA AUGMENTATION (Research-Grade Pipeline)
# ═══════════════════════════════════════════════════════════════════════════════
#
# Augmentation strategy for deepfake detection:
#   1. Geometric: HorizontalFlip, ShiftScaleRotate (face alignment variations)
#   2. Color: Brightness, Contrast, Hue, RGBShift (lighting/color manipulation)
#   3. Compression: JPEG artifacts, Downscaling (social media compression)
#   4. Noise: Gaussian noise, blur (camera quality variations)
#   5. Dropout: CoarseDropout (occlusion robustness)
#
# ═══════════════════════════════════════════════════════════════════════════════

# ImageNet normalization (required for pretrained EfficientNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def get_train_transforms():
    """
    Training augmentations optimized for deepfake detection.
    
    Key considerations:
      - Preserve facial structure (mild geometric transforms)
      - Simulate social media compression (JPEG, downscaling)
      - Color variations (lighting, camera differences)
      - Noise/blur (video quality variations)
    """
    return A.Compose([
        # Geometric (mild - preserve facial structure)
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05, 
            scale_limit=0.05, 
            rotate_limit=10, 
            border_mode=cv2.BORDER_REFLECT_101,
            p=0.3
        ),
        
        # Color augmentations (simulate different cameras/lighting)
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.3),
        A.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15, p=0.3),
        
        # Compression artifacts (critical for deepfake robustness)
        A.ImageCompression(quality_lower=50, quality_upper=100, p=0.5),
        A.Downscale(scale_min=0.5, scale_max=0.9, p=0.3),  # Simulate low-res uploads
        
        # Noise and blur (camera quality variations)
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        
        # Occlusion (partial face visibility)
        A.CoarseDropout(
            max_holes=4, 
            max_height=20, 
            max_width=20, 
            fill_value=0,
            p=0.2
        ),
        
        # Normalize and convert
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def get_val_transforms():
    """Validation transforms: only normalization (no augmentation)."""
    return A.Compose([
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


print("✓ Augmentation pipelines defined")
print("  Train augmentations:")
print("    - Geometric: HorizontalFlip, ShiftScaleRotate")
print("    - Color: Brightness/Contrast, HSV, RGBShift")
print("    - Compression: JPEG, Downscale")
print("    - Noise: GaussNoise, GaussianBlur")
print("    - Occlusion: CoarseDropout")
print("  Val: Normalize only (no augmentation)")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PYTORCH DATASETS (Frame-Level + Video-Level for Temporal Aggregation)
# ═══════════════════════════════════════════════════════════════════════════════

class DeepfakeFaceDataset(Dataset):
    """
    PyTorch Dataset for frame-level deepfake detection.
    Each sample is a single face crop with its video-level label.
    Used for: Baseline frame-level training (without temporal modeling)
    """

    def __init__(self, videos: List[dict], face_data: Dict[str, List[np.ndarray]],
                 transform=None, is_train=True):
        self.transform = transform
        self.is_train = is_train

        # Build flat list of (face_crop, label, video_id)
        self.samples = []
        for video in videos:
            video_id = video['video_id']
            label = video['label']

            if video_id in face_data:
                for face in face_data[video_id]:
                    self.samples.append((face, label, video_id))

        print(f"  Frame-level dataset: {len(self.samples)} samples from {len(videos)} videos")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        face, label, video_id = self.samples[idx]

        if self.transform:
            augmented = self.transform(image=face)
            face = augmented['image']

        return {
            'image': face,
            'label': torch.tensor(label, dtype=torch.float32),
            'video_id': video_id
        }


class DeepfakeVideoDataset(Dataset):
    """
    PyTorch Dataset for VIDEO-LEVEL deepfake detection with temporal modeling.
    Each sample is ALL frames from one video, enabling BiLSTM/Transformer to
    detect inter-frame inconsistencies (temporal flickering, blending shifts).

    This is the KEY UPGRADE for research-grade temporal analysis.

    Returns:
        frames: Tensor of shape (T, C, H, W) where T = num frames
        label: Video-level label (0 = real, 1 = fake)
        video_id: String identifier for Late Fusion
        mask: Boolean tensor indicating valid frames (for variable-length sequences)
    """

    def __init__(self, videos: List[dict], face_data: Dict[str, List[np.ndarray]],
                 transform=None, max_frames: int = 15, is_train: bool = True):
        self.transform = transform
        self.max_frames = max_frames
        self.is_train = is_train

        # Filter videos that have face data
        self.videos = []
        for video in videos:
            video_id = video['video_id']
            if video_id in face_data and len(face_data[video_id]) >= 3:
                self.videos.append({
                    'video_id': video_id,
                    'label': video['label'],
                    'faces': face_data[video_id]
                })

        print(f"  Video-level dataset: {len(self.videos)} videos (for temporal aggregation)")

    def __len__(self):
        return len(self.videos)

    def __getitem__(self, idx):
        video = self.videos[idx]
        faces = video['faces']
        label = video['label']
        video_id = video['video_id']

        # Ensure exactly max_frames (pad or truncate)
        if len(faces) >= self.max_frames:
            # Sample evenly if more frames available
            indices = np.linspace(0, len(faces) - 1, self.max_frames, dtype=int)
            selected_faces = [faces[i] for i in indices]
            mask = torch.ones(self.max_frames, dtype=torch.bool)
        else:
            # Pad with last frame if fewer frames
            selected_faces = faces.copy()
            while len(selected_faces) < self.max_frames:
                selected_faces.append(faces[-1])  # Repeat last frame
            mask = torch.zeros(self.max_frames, dtype=torch.bool)
            mask[:len(faces)] = True

        # Apply transforms to each frame
        frame_tensors = []
        for face in selected_faces:
            if self.transform:
                augmented = self.transform(image=face)
                frame_tensor = augmented['image']
            else:
                frame_tensor = torch.from_numpy(face.transpose(2, 0, 1)).float() / 255.0
            frame_tensors.append(frame_tensor)

        # Stack into (T, C, H, W)
        frames = torch.stack(frame_tensors, dim=0)

        return {
            'frames': frames,  # (T, C, H, W)
            'label': torch.tensor(label, dtype=torch.float32),
            'video_id': video_id,
            'mask': mask  # (T,) boolean mask for valid frames
        }


# Create VIDEO-LEVEL datasets for temporal modeling
print("\nCreating VIDEO-LEVEL datasets for temporal aggregation...")
train_dataset = DeepfakeVideoDataset(
    train_videos, face_data, transform=get_train_transforms(),
    max_frames=cfg.FRAMES_PER_VIDEO, is_train=True
)
val_dataset = DeepfakeVideoDataset(
    val_videos, face_data, transform=get_val_transforms(),
    max_frames=cfg.FRAMES_PER_VIDEO, is_train=False
)

# Create dataloaders (video-level batching)
train_loader = DataLoader(
    train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True,
    num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False,
    num_workers=cfg.NUM_WORKERS, pin_memory=True
)

print(f"\n✓ Train loader: {len(train_loader)} batches (video-level)")
print(f"✓ Val loader: {len(val_loader)} batches (video-level)")
print(f"✓ Each batch: {cfg.BATCH_SIZE} videos × {cfg.FRAMES_PER_VIDEO} frames = {cfg.BATCH_SIZE * cfg.FRAMES_PER_VIDEO} images")

## 4. CNN Model Architecture

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SPATIO-TEMPORAL CNN MODEL (EfficientNet-B4 + BiLSTM Temporal Aggregation)
# ═══════════════════════════════════════════════════════════════════════════════
#
# RESEARCH CONTRIBUTION: Temporal modeling for deepfake detection
#
# Key Innovation: Unlike naive frame averaging (np.mean(probs)), we use a BiLSTM
# with Multi-Head Attention to model temporal dependencies between frames.
# This enables detection of inter-frame artifacts:
#   - Temporal flickering (GAN sampling noise)
#   - Blending boundary shifts (face swap edge inconsistencies)
#   - Unnatural motion patterns (physics-defying movements)
#   - Phase inconsistencies (audio-visual desync markers)
#
# Architecture:
#   Video (T frames) → EfficientNet-B4 (spatial) → T × 1792-dim vectors
#                    → BiLSTM (temporal) → T × 512-dim vectors
#                    → Multi-Head Self-Attention → Weighted temporal features
#                    → Classifier → 1 logit (deepfake probability)
#
# Memory Optimization: Gradient checkpointing for P100 (16GB VRAM)
#
# ═══════════════════════════════════════════════════════════════════════════════

from torch.utils.checkpoint import checkpoint


class TemporalAttention(nn.Module):
    """
    Multi-Head Self-Attention for temporal feature aggregation.
    
    Learns which frames are most informative for deepfake detection,
    allowing the model to focus on frames with suspicious artifacts.
    """

    def __init__(self, feature_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.attention = nn.MultiheadAttention(
            embed_dim=feature_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.layer_norm = nn.LayerNorm(feature_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Args:
            x: (B, T, D) temporal features
            mask: (B, T) boolean mask (True = valid, False = padding)
        Returns:
            pooled: (B, D) aggregated features
            attn_weights: (B, T, T) attention weights for visualization
        """
        # Self-attention with residual connection
        key_padding_mask = ~mask if mask is not None else None
        attn_out, attn_weights = self.attention(x, x, x, key_padding_mask=key_padding_mask)
        x = self.layer_norm(x + self.dropout(attn_out))

        # Weighted average pooling using mask
        if mask is not None:
            mask_expanded = mask.unsqueeze(-1).float()  # (B, T, 1)
            x = x * mask_expanded
            pooled = x.sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)
        else:
            pooled = x.mean(dim=1)

        return pooled, attn_weights


class SpatioTemporalDeepfakeCNN(nn.Module):
    """
    Spatio-Temporal Deepfake Detector for research-grade video analysis.

    This model extracts spatial features from each frame using EfficientNet-B4,
    then uses a BiLSTM with Multi-Head Attention to model temporal dependencies
    across the video sequence, enabling detection of inter-frame artifacts.

    Key Features:
      - EfficientNet-B4 backbone (1792-dim features)
      - 2-layer BiLSTM (256 hidden → 512-dim output)
      - 4-head self-attention for temporal pooling
      - Gradient checkpointing for P100 memory efficiency
    """

    def __init__(
        self,
        model_name: str = 'efficientnet_b4',
        hidden_dim: int = 256,
        dropout: float = 0.4,
        pretrained: bool = True,
        temporal_type: str = 'bilstm_attention',
        lstm_hidden: int = 256,
        lstm_layers: int = 2,
        attention_heads: int = 4,
        freeze_backbone: bool = False,
        use_gradient_checkpointing: bool = True
    ):
        super().__init__()

        self.temporal_type = temporal_type
        self.use_gradient_checkpointing = use_gradient_checkpointing

        # ─── Spatial Feature Extractor (EfficientNet-B4) ───────────────────────
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,  # Remove classifier head
            global_pool='avg'
        )
        self.backbone_dim = self.backbone.num_features  # 1792 for EfficientNet-B4

        # Enable gradient checkpointing for memory efficiency on P100
        if use_gradient_checkpointing and hasattr(self.backbone, 'set_grad_checkpointing'):
            self.backbone.set_grad_checkpointing(enable=True)
            print(f"  ⚡ Gradient checkpointing enabled (saves ~40% VRAM)")

        # Optionally freeze backbone for faster training
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
            print(f"  ❄ Backbone frozen (faster training, less memory)")

        # ─── Temporal Feature Aggregator ───────────────────────────────────────
        if temporal_type == 'bilstm':
            self.temporal = nn.LSTM(
                input_size=self.backbone_dim,
                hidden_size=lstm_hidden,
                num_layers=lstm_layers,
                batch_first=True,
                bidirectional=True,
                dropout=dropout if lstm_layers > 1 else 0
            )
            temporal_out_dim = lstm_hidden * 2  # Bidirectional

        elif temporal_type == 'bilstm_attention':
            self.temporal = nn.LSTM(
                input_size=self.backbone_dim,
                hidden_size=lstm_hidden,
                num_layers=lstm_layers,
                batch_first=True,
                bidirectional=True,
                dropout=dropout if lstm_layers > 1 else 0
            )
            self.temporal_attention = TemporalAttention(
                feature_dim=lstm_hidden * 2,
                num_heads=attention_heads,
                dropout=dropout
            )
            temporal_out_dim = lstm_hidden * 2

        elif temporal_type == 'transformer':
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=self.backbone_dim,
                nhead=attention_heads,
                dim_feedforward=self.backbone_dim * 2,
                dropout=dropout,
                activation='gelu',
                batch_first=True
            )
            self.temporal = nn.TransformerEncoder(encoder_layer, num_layers=lstm_layers)
            self.temporal_attention = TemporalAttention(
                feature_dim=self.backbone_dim,
                num_heads=attention_heads,
                dropout=dropout
            )
            temporal_out_dim = self.backbone_dim

        else:
            raise ValueError(f"Unknown temporal_type: {temporal_type}")

        self.temporal_out_dim = temporal_out_dim

        # ─── Classification Head ───────────────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Linear(temporal_out_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, 1)
        )

        # Initialize classifier weights (Xavier for better gradient flow)
        self._init_weights()

        print(f"\n✓ SpatioTemporalDeepfakeCNN initialized")
        print(f"  Backbone: {model_name} ({self.backbone_dim}-dim features)")
        print(f"  Temporal: {temporal_type} (LSTM hidden: {lstm_hidden}, layers: {lstm_layers})")
        print(f"  Attention heads: {attention_heads}")
        print(f"  Classifier: {temporal_out_dim} → {hidden_dim} → {hidden_dim // 2} → 1")

    def _init_weights(self):
        """Initialize classifier weights using Xavier uniform."""
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def _extract_single_frame_features(self, frame):
        """Helper for gradient checkpointing."""
        return self.backbone(frame)

    def extract_frame_features(self, frames):
        """
        Extract spatial features from all frames.

        Args:
            frames: (B, T, C, H, W) batch of video frames

        Returns:
            (B, T, D) frame-level features
        """
        B, T, C, H, W = frames.shape

        # Reshape to process all frames through backbone: (B*T, C, H, W)
        frames_flat = frames.view(B * T, C, H, W)

        # Extract features (with gradient checkpointing if enabled)
        if self.use_gradient_checkpointing and self.training:
            # Process in smaller chunks to save memory
            chunk_size = max(1, (B * T) // 4)  # 4 chunks
            features_list = []
            for i in range(0, B * T, chunk_size):
                chunk = frames_flat[i:i + chunk_size]
                chunk_features = checkpoint(
                    self._extract_single_frame_features, 
                    chunk,
                    use_reentrant=False
                )
                features_list.append(chunk_features)
            features_flat = torch.cat(features_list, dim=0)
        else:
            features_flat = self.backbone(frames_flat)  # (B*T, D)

        # Reshape back to sequence: (B, T, D)
        features = features_flat.view(B, T, -1)

        return features

    def forward(self, frames, mask=None):
        """
        Forward pass for video-level classification.

        Args:
            frames: (B, T, C, H, W) batch of video frames
            mask: (B, T) boolean mask for valid frames

        Returns:
            logits: (B,) deepfake logits
        """
        # Extract spatial features from each frame
        features = self.extract_frame_features(frames)  # (B, T, D)

        # Apply temporal modeling
        if self.temporal_type == 'bilstm':
            lstm_out, _ = self.temporal(features)  # (B, T, 2*H)
            # Masked mean pooling
            if mask is not None:
                mask_expanded = mask.unsqueeze(-1).float()
                pooled = (lstm_out * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)
            else:
                pooled = lstm_out.mean(dim=1)

        elif self.temporal_type == 'bilstm_attention':
            lstm_out, _ = self.temporal(features)  # (B, T, 2*H)
            pooled, _ = self.temporal_attention(lstm_out, mask)  # (B, 2*H)

        elif self.temporal_type == 'transformer':
            attn_mask = ~mask if mask is not None else None
            transformer_out = self.temporal(features, src_key_padding_mask=attn_mask)
            pooled, _ = self.temporal_attention(transformer_out, mask)

        # Classify
        logits = self.classifier(pooled).squeeze(-1)  # (B,)

        return logits

    def get_attention_weights(self, frames, mask=None):
        """
        Get temporal attention weights for visualization.
        Shows which frames the model focuses on for classification.
        """
        with torch.no_grad():
            features = self.extract_frame_features(frames)

            if self.temporal_type == 'bilstm_attention':
                lstm_out, _ = self.temporal(features)
                _, attn_weights = self.temporal_attention(lstm_out, mask)
                return attn_weights
            elif self.temporal_type == 'transformer':
                attn_mask = ~mask if mask is not None else None
                transformer_out = self.temporal(features, src_key_padding_mask=attn_mask)
                _, attn_weights = self.temporal_attention(transformer_out, mask)
                return attn_weights
            else:
                return None


# ═══════════════════════════════════════════════════════════════════════════════
# LEGACY: Frame-Level Model (for Ablation Study)
# ═══════════════════════════════════════════════════════════════════════════════

class DeepfakeCNN(nn.Module):
    """
    BASELINE: Simple frame-level EfficientNet classifier.
    
    Kept for ablation studies comparing:
      - Frame averaging (this) vs Temporal modeling (SpatioTemporalDeepfakeCNN)
    
    This model does NOT capture inter-frame temporal patterns.
    """

    def __init__(self, model_name='efficientnet_b4', hidden_dim=256, dropout=0.4, pretrained=True):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool='avg'
        )
        backbone_dim = self.backbone.num_features

        self.classifier = nn.Sequential(
            nn.Linear(backbone_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits.squeeze(-1)


# ═══════════════════════════════════════════════════════════════════════════════
# CREATE MODEL
# ═══════════════════════════════════════════════════════════════════════════════

model = SpatioTemporalDeepfakeCNN(
    model_name=cfg.MODEL_NAME,
    hidden_dim=cfg.HIDDEN_DIM,
    dropout=cfg.DROPOUT,
    pretrained=True,
    temporal_type=cfg.TEMPORAL_TYPE,
    lstm_hidden=cfg.LSTM_HIDDEN,
    lstm_layers=cfg.LSTM_LAYERS,
    attention_heads=cfg.ATTENTION_HEADS,
    freeze_backbone=cfg.FREEZE_BACKBONE,
    use_gradient_checkpointing=cfg.USE_GRADIENT_CHECKPOINTING
).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
backbone_params = sum(p.numel() for p in model.backbone.parameters())
temporal_params = total_params - backbone_params

print(f"\n{'='*70}")
print("MODEL PARAMETERS")
print(f"{'='*70}")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Backbone (spatial):   {backbone_params:,} ({backbone_params/1e6:.1f}M)")
print(f"  Temporal + Classifier: {temporal_params:,} ({temporal_params/1e6:.1f}M)")

# Estimate memory usage
param_memory = total_params * 4 / (1024**3)  # float32
grad_memory = trainable_params * 4 / (1024**3)
batch_memory = cfg.BATCH_SIZE * cfg.FRAMES_PER_VIDEO * 3 * 224 * 224 * 4 / (1024**3)
total_memory = param_memory + grad_memory + batch_memory * 3  # rough estimate

print(f"\n  Estimated VRAM usage:")
print(f"    Parameters: ~{param_memory:.1f} GB")
print(f"    Gradients:  ~{grad_memory:.1f} GB")
print(f"    Batch data: ~{batch_memory:.1f} GB")
print(f"    Total:      ~{total_memory:.1f} GB (P100 has 16 GB)")
print(f"{'='*70}")


## 5. Training Protocol

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING UTILITIES (Video-Level with Temporal Modeling)
# ═══════════════════════════════════════════════════════════════════════════════

def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    """
    Train for one epoch with video-level batching and AMP.

    Key difference from frame-level: Each batch contains full videos (T frames each),
    and the model processes all frames through the BiLSTM temporal aggregator.
    """
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    num_samples = 0

    pbar = tqdm(loader, desc="Training", leave=False)

    for batch in pbar:
        # Video-level batch: (B, T, C, H, W)
        frames = batch['frames'].to(device)
        labels = batch['label'].to(device)
        mask = batch['mask'].to(device)

        optimizer.zero_grad()

        # AMP forward pass
        amp_context = autocast(device_type=str(device).split(":")[0]) if USE_NEW_AMP else autocast()
        with amp_context:
            logits = model(frames, mask)  # (B,) - one logit per video
            loss = criterion(logits, labels)

        # AMP backward pass
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        # Track metrics (video-level)
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        num_samples += batch_size

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_preds.extend(probs)
        all_labels.extend(labels.cpu().numpy())

        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / num_samples

    # Calculate metrics
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc = accuracy_score(all_labels, (all_preds > 0.5).astype(int))

    if len(np.unique(all_labels)) > 1:
        auc = roc_auc_score(all_labels, all_preds)
    else:
        auc = 0.5

    return avg_loss, acc, auc


@torch.no_grad()
def validate(model, loader, criterion, device):
    """
    Validate the model with video-level batching.
    """
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    num_samples = 0

    for batch in tqdm(loader, desc="Validating", leave=False):
        frames = batch['frames'].to(device)
        labels = batch['label'].to(device)
        mask = batch['mask'].to(device)

        amp_context = autocast(device_type=str(device).split(":")[0]) if USE_NEW_AMP else autocast()
        with amp_context:
            logits = model(frames, mask)
            loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        num_samples += batch_size

        probs = torch.sigmoid(logits).cpu().numpy()
        all_preds.extend(probs)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / num_samples

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc = accuracy_score(all_labels, (all_preds > 0.5).astype(int))

    if len(np.unique(all_labels)) > 1:
        auc = roc_auc_score(all_labels, all_preds)
    else:
        auc = 0.5

    f1 = f1_score(all_labels, (all_preds > 0.5).astype(int))

    return avg_loss, acc, auc, f1


print("✓ Video-level training utilities defined")
print("  - Each batch processes full videos through BiLSTM temporal aggregator")
print("  - AMP mixed precision enabled for memory efficiency")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING LOOP
# ═══════════════════════════════════════════════════════════════════════════════

# Loss, optimizer, scheduler
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=cfg.LEARNING_RATE, 
    weight_decay=cfg.WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.NUM_EPOCHS, eta_min=1e-6
)

# AMP scaler for mixed precision
scaler = GradScaler("cuda") if (USE_NEW_AMP and torch.cuda.is_available()) else GradScaler()

# Training history
history = {
    'train_loss': [], 'train_acc': [], 'train_auc': [],
    'val_loss': [], 'val_acc': [], 'val_auc': [], 'val_f1': []
}

best_val_auc = 0.0
best_epoch = 0
patience = 5
patience_counter = 0

print("\n" + "="*70)
print("TRAINING CNN SPATIAL STREAM")
print("="*70)
print(f"  Epochs: {cfg.NUM_EPOCHS}")
print(f"  Batch size: {cfg.BATCH_SIZE}")
print(f"  Learning rate: {cfg.LEARNING_RATE}")
print(f"  Mixed Precision: Enabled (AMP)")
print("="*70 + "\n")

for epoch in range(cfg.NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{cfg.NUM_EPOCHS}")
    print("-" * 40)
    
    # Train
    train_loss, train_acc, train_auc = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, DEVICE
    )
    
    # Validate
    val_loss, val_acc, val_auc, val_f1 = validate(
        model, val_loader, criterion, DEVICE
    )
    
    # Update scheduler
    scheduler.step()
    
    # Record history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_auc'].append(train_auc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)
    history['val_f1'].append(val_f1)
    
    # Print metrics
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | AUC: {train_auc:.4f}")
    print(f"  Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | AUC: {val_auc:.4f} | F1: {val_f1:.4f}")
    
    # Save best model + early stopping
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_epoch = epoch + 1
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(cfg.OUTPUT_DIR, "best_cnn_model.pth"))
        print(f"  ★ New best model saved (AUC: {val_auc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"  Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
            break

print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)
print(f"  Best Epoch: {best_epoch}")
print(f"  Best Val AUC: {best_val_auc:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING CURVES
# ═══════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train', marker='o')
axes[0].plot(history['val_loss'], label='Val', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train', marker='o')
axes[1].plot(history['val_acc'], label='Val', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# AUC
axes[2].plot(history['train_auc'], label='Train', marker='o')
axes[2].plot(history['val_auc'], label='Val', marker='s')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('AUC')
axes[2].set_title('AUC Curve')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Video-Level Inference & Export

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LOAD BEST MODEL
# ═══════════════════════════════════════════════════════════════════════════════

# Load best model weights
model.load_state_dict(torch.load(os.path.join(cfg.OUTPUT_DIR, "best_cnn_model.pth"), map_location=DEVICE, weights_only=True))
model.eval()
print("✓ Best model loaded for inference")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VIDEO-LEVEL PREDICTION (Using Temporal Model)
# ═══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def predict_video_temporal(model, faces: List[np.ndarray], transform, device, max_frames: int = 15) -> Tuple[float, Optional[np.ndarray]]:
    """
    Predict deepfake probability for a single video using temporal model.

    Unlike frame averaging, this passes ALL frames through the BiLSTM to capture
    inter-frame temporal patterns (flickering, blending shifts, motion anomalies).

    Args:
        model: SpatioTemporalDeepfakeCNN model
        faces: List of face crops (numpy arrays)
        transform: Albumentations transform
        device: torch device
        max_frames: Maximum frames to use

    Returns:
        P_CNN: Deepfake probability (0 = real, 1 = fake)
        attn_weights: Temporal attention weights (which frames mattered most)
    """
    model.eval()

    if len(faces) == 0:
        return 0.5, None

    # Prepare frames (same logic as dataset)
    if len(faces) >= max_frames:
        indices = np.linspace(0, len(faces) - 1, max_frames, dtype=int)
        selected_faces = [faces[i] for i in indices]
        mask = torch.ones(max_frames, dtype=torch.bool)
    else:
        selected_faces = faces.copy()
        while len(selected_faces) < max_frames:
            selected_faces.append(faces[-1])
        mask = torch.zeros(max_frames, dtype=torch.bool)
        mask[:len(faces)] = True

    # Transform frames
    frame_tensors = []
    for face in selected_faces:
        augmented = transform(image=face)
        frame_tensors.append(augmented['image'])

    # Stack: (T, C, H, W) -> (1, T, C, H, W)
    frames = torch.stack(frame_tensors, dim=0).unsqueeze(0).to(device)
    mask = mask.unsqueeze(0).to(device)

    # Forward pass with temporal modeling
    amp_context = autocast(device_type=str(device).split(":")[0]) if USE_NEW_AMP else autocast()
    with amp_context:
        logit = model(frames, mask)

    prob = torch.sigmoid(logit).item()

    # Get attention weights (for visualization)
    try:
        attn_weights = model.get_attention_weights(frames, mask)
        if attn_weights is not None:
            attn_weights = attn_weights.cpu().numpy()
    except Exception:
        attn_weights = None

    return prob, attn_weights


def generate_video_predictions(model, all_videos, face_data, device):
    """
    Generate video-level predictions for all videos using temporal model.
    """
    transform = get_val_transforms()

    results = []

    print("\n" + "="*70)
    print("GENERATING VIDEO-LEVEL PREDICTIONS (Temporal Model)")
    print("="*70)

    for video in tqdm(all_videos, desc="Predicting"):
        video_id = video['video_id']

        if video_id in face_data:
            faces = face_data[video_id]
            p_cnn, _ = predict_video_temporal(model, faces, transform, device, cfg.FRAMES_PER_VIDEO)
        else:
            p_cnn = 0.5  # Neutral if no face data

        results.append({
            'video_id': video_id,
            'P_CNN': p_cnn
        })

    return pd.DataFrame(results)


# Generate predictions
predictions_df = generate_video_predictions(model, all_videos, face_data, DEVICE)

print(f"\n✓ Predictions generated for {len(predictions_df)} videos")
print(f"✓ Using temporal model (BiLSTM + Attention) for inter-frame analysis")
print(predictions_df.head(10))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPORT PREDICTIONS
# ═══════════════════════════════════════════════════════════════════════════════

# Add ground truth labels
video_labels = {v['video_id']: v['label'] for v in all_videos}
predictions_df['true_label'] = predictions_df['video_id'].map(video_labels)

# Save predictions CSV (with true_label to match rPPG export format)
csv_path = os.path.join(cfg.OUTPUT_DIR, "cnn_predictions.csv")
predictions_df.to_csv(csv_path, index=False)
print(f"\n✓ Predictions saved to: {csv_path}")

# Show distribution
print("\nP_CNN Distribution:")
print(predictions_df['P_CNN'].describe())

# Alias for metrics below
predictions_df['label'] = predictions_df['true_label']

# Calculate video-level metrics
y_true = predictions_df['label'].values
y_pred_proba = predictions_df['P_CNN'].values
y_pred = (y_pred_proba > 0.5).astype(int)

video_acc = accuracy_score(y_true, y_pred)
# Handle edge case where only one class is present
if len(np.unique(y_true)) > 1:
    video_auc = roc_auc_score(y_true, y_pred_proba)
else:
    video_auc = 0.5
    print("Warning: Only one class in predictions, AUC set to 0.5")
video_f1 = f1_score(y_true, y_pred)

print("\n" + "="*70)
print("VIDEO-LEVEL METRICS (Full Dataset)")
print("="*70)
print(f"  Accuracy: {video_acc:.4f}")
print(f"  AUC-ROC:  {video_auc:.4f}")
print(f"  F1-Score: {video_f1:.4f}")
print("="*70)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VISUALIZE PREDICTION DISTRIBUTION
# ═══════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution by class
real_preds = predictions_df[predictions_df['label'] == 0]['P_CNN']
fake_preds = predictions_df[predictions_df['label'] == 1]['P_CNN']

axes[0].hist(real_preds, bins=30, alpha=0.6, label='Real', color='green', density=True)
axes[0].hist(fake_preds, bins=30, alpha=0.6, label='Fake', color='red', density=True)
axes[0].axvline(x=0.5, color='black', linestyle='--', label='Threshold')
axes[0].set_xlabel('P_CNN')
axes[0].set_ylabel('Density')
axes[0].set_title('P_CNN Distribution by Class')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ROC Curve
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
axes[1].plot(fpr, tpr, 'b-', label=f'CNN (AUC = {video_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve (Video-Level)')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, 'cnn_results.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# GRAD-CAM VISUALIZATION (Research Interpretability)
# ═══════════════════════════════════════════════════════════════════════════════
#
# Grad-CAM (Gradient-weighted Class Activation Mapping) generates heatmaps showing
# which spatial regions of the face triggered the deepfake prediction.
#
# This is ESSENTIAL for research papers:
#   1. Proves the model learns meaningful artifacts (jawline blending, eye artifacts)
#   2. Enables qualitative analysis in methodology section
#   3. Addresses reviewer concerns about overfitting to background noise
#
# ═══════════════════════════════════════════════════════════════════════════════

class GradCAM:
    """
    Grad-CAM implementation for EfficientNet backbone visualization.

    Generates heatmaps showing which regions of the face contributed most
    to the deepfake classification decision.
    """

    def __init__(self, model, target_layer=None):
        """
        Args:
            model: SpatioTemporalDeepfakeCNN model
            target_layer: Layer to compute Grad-CAM on (defaults to last conv layer)
        """
        self.model = model
        self.gradients = None
        self.activations = None

        # Get target layer (last convolutional layer of EfficientNet)
        if target_layer is None:
            # EfficientNet-B4: conv_head is the last conv layer before pooling
            self.target_layer = model.backbone.conv_head
        else:
            self.target_layer = target_layer

        # Register hooks
        self.target_layer.register_forward_hook(self._save_activation)
        self.target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, frame: torch.Tensor, class_idx: int = 1) -> np.ndarray:
        """
        Generate Grad-CAM heatmap for a single frame.

        Args:
            frame: Single frame tensor (1, C, H, W)
            class_idx: Target class (1 = fake, 0 = real)

        Returns:
            Heatmap normalized to [0, 1] with same spatial size as input
        """
        self.model.eval()

        # Forward pass through backbone only (for single frame)
        features = self.model.backbone(frame)  # (1, D)

        # Create a simple classifier output for gradient computation
        # We use the model's classifier on the single frame features
        # Note: This is a simplified approach for visualization
        B, T = 1, 1  # Single frame

        # For visualization, we bypass temporal modeling and directly classify
        # the frame-level features
        logits = self.model.classifier(features).squeeze(-1)

        # Backward pass
        self.model.zero_grad()
        if class_idx == 1:
            # Gradient for "fake" prediction
            logits.backward(retain_graph=True)
        else:
            # Gradient for "real" prediction (negate)
            (-logits).backward(retain_graph=True)

        # Compute Grad-CAM
        gradients = self.gradients  # (1, C, H', W')
        activations = self.activations  # (1, C, H', W')

        # Global average pooling of gradients
        weights = gradients.mean(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)

        # Weighted combination of activation maps
        cam = (weights * activations).sum(dim=1, keepdim=True)  # (1, 1, H', W')

        # ReLU to keep only positive contributions
        cam = F.relu(cam)

        # Normalize
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        # Resize to original image size
        cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)

        return cam.squeeze().cpu().numpy()


def visualize_gradcam(model, face: np.ndarray, transform, device, save_path: str = None):
    """
    Generate and display Grad-CAM visualization for a single face.

    Args:
        model: Trained model
        face: Face crop as numpy array (H, W, 3)
        transform: Validation transform
        device: torch device
        save_path: Optional path to save the figure
    """
    # Initialize Grad-CAM
    gradcam = GradCAM(model)

    # Prepare input
    augmented = transform(image=face)
    frame_tensor = augmented['image'].unsqueeze(0).to(device)

    # Generate heatmap
    heatmap = gradcam.generate(frame_tensor, class_idx=1)

    # Get prediction
    with torch.no_grad():
        # Simple forward through backbone + classifier for single frame
        features = model.backbone(frame_tensor)
        logit = model.classifier(features).squeeze()
        prob = torch.sigmoid(logit).item()

    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Original face
    axes[0].imshow(face)
    axes[0].set_title('Original Face')
    axes[0].axis('off')

    # Heatmap
    axes[1].imshow(heatmap, cmap='jet')
    axes[1].set_title('Grad-CAM Heatmap')
    axes[1].axis('off')

    # Overlay
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(face, 0.5, heatmap_colored, 0.5, 0)
    axes[2].imshow(overlay)
    axes[2].set_title(f'Overlay (P_fake = {prob:.3f})')
    axes[2].axis('off')

    plt.suptitle('Grad-CAM: Where the model looks to detect deepfakes', fontsize=14)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

    plt.show()

    return heatmap, prob


def generate_gradcam_gallery(model, face_data, all_videos, device, n_samples: int = 8):
    """
    Generate Grad-CAM gallery showing model attention on real vs fake faces.

    This is essential for research paper methodology section.
    """
    transform = get_val_transforms()

    # Select samples
    real_samples = [(v['video_id'], face_data[v['video_id']][0])
                    for v in all_videos if v['label'] == 0 and v['video_id'] in face_data][:n_samples//2]
    fake_samples = [(v['video_id'], face_data[v['video_id']][0])
                    for v in all_videos if v['label'] == 1 and v['video_id'] in face_data][:n_samples//2]

    all_samples = real_samples + fake_samples
    labels = [0] * len(real_samples) + [1] * len(fake_samples)

    # Generate gallery
    n_cols = 4
    n_rows = len(all_samples) // n_cols + (1 if len(all_samples) % n_cols else 0)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes.flatten()

    gradcam = GradCAM(model)

    for idx, ((video_id, face), label) in enumerate(zip(all_samples, labels)):
        if idx >= len(axes):
            break

        # Prepare input
        augmented = transform(image=face)
        frame_tensor = augmented['image'].unsqueeze(0).to(device)

        # Generate heatmap
        try:
            heatmap = gradcam.generate(frame_tensor, class_idx=1)

            # Get prediction
            with torch.no_grad():
                features = model.backbone(frame_tensor)
                logit = model.classifier(features).squeeze()
                prob = torch.sigmoid(logit).item()

            # Create overlay
            heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
            heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
            overlay = cv2.addWeighted(face, 0.5, heatmap_colored, 0.5, 0)

            # Display
            axes[idx].imshow(overlay)
            label_str = 'REAL' if label == 0 else 'FAKE'
            pred_str = 'Real' if prob < 0.5 else 'Fake'
            color = 'green' if (prob < 0.5) == (label == 0) else 'red'
            axes[idx].set_title(f'{label_str} → {pred_str} (P={prob:.2f})', color=color, fontsize=10)
            axes[idx].axis('off')

        except Exception as e:
            axes[idx].text(0.5, 0.5, f'Error: {str(e)[:20]}', ha='center', va='center')
            axes[idx].axis('off')

    # Hide unused axes
    for idx in range(len(all_samples), len(axes)):
        axes[idx].axis('off')

    plt.suptitle('Grad-CAM Visualization: Model Attention on Real vs Fake Faces', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.OUTPUT_DIR, 'gradcam_gallery.png'), dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n✓ Grad-CAM gallery saved to: {cfg.OUTPUT_DIR}/gradcam_gallery.png")


# Generate Grad-CAM gallery
print("\n" + "="*70)
print("GRAD-CAM VISUALIZATION (Model Interpretability)")
print("="*70)
print("  Generating attention heatmaps for research paper...")

try:
    generate_gradcam_gallery(model, face_data, all_videos, DEVICE, n_samples=8)
    print("  ✓ Heatmaps show which facial regions triggered deepfake detection")
    print("  ✓ Use these visualizations in your paper's methodology section")
except Exception as e:
    print(f"  ⚠ Grad-CAM visualization failed: {e}")
    print("  This is optional - model training/inference still works")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SAVE ALL ARTIFACTS
# ═══════════════════════════════════════════════════════════════════════════════

# Save final model weights
torch.save(model.state_dict(), os.path.join(cfg.OUTPUT_DIR, "cnn_spatial_stream_final.pth"))

# Save training history
history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(cfg.OUTPUT_DIR, "cnn_training_history.csv"), index=False)

# Save config
config_dict = {
    'model_name': cfg.MODEL_NAME,
    'hidden_dim': cfg.HIDDEN_DIM,
    'dropout': cfg.DROPOUT,
    'img_size': cfg.IMG_SIZE,
    'frames_per_video': cfg.FRAMES_PER_VIDEO,
    'batch_size': cfg.BATCH_SIZE,
    'learning_rate': cfg.LEARNING_RATE,
    'num_epochs': cfg.NUM_EPOCHS,
    'best_val_auc': best_val_auc,
    'video_level_auc': video_auc,
}
pd.DataFrame([config_dict]).to_csv(os.path.join(cfg.OUTPUT_DIR, "cnn_config.csv"), index=False)

print("\n" + "="*70)
print("ALL ARTIFACTS SAVED")
print("="*70)
print(f"  ✓ cnn_predictions.csv        (video-level P_CNN scores)")
print(f"  ✓ best_cnn_model.pth          (best model weights)")
print(f"  ✓ cnn_spatial_stream_final.pth (final model weights)")
print(f"  ✓ cnn_training_history.csv   (training metrics)")
print(f"  ✓ cnn_config.csv             (configuration)")
print(f"  ✓ training_curves.png        (loss/acc/auc plots)")
print(f"  ✓ cnn_results.png            (prediction distribution)")
print("="*70)

## 7. Late Fusion Integration Guide

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LATE FUSION INTEGRATION GUIDE
# ═══════════════════════════════════════════════════════════════════════════════

print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                       LATE FUSION INTEGRATION GUIDE                          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  This notebook outputs: cnn_predictions.csv                                  ║
║  Columns: video_id, P_CNN                                                    ║
║                                                                              ║
║  To fuse with rPPG predictions from 2ND_MODEL.ipynb:                        ║
║                                                                              ║
║  ┌─────────────────────────────────────────────────────────────────────────┐ ║
║  │  # Load both predictions                                                │ ║
║  │  cnn_df = pd.read_csv('cnn_predictions.csv')                           │ ║
║  │  rppg_df = pd.read_csv('rppg_predictions.csv')                         │ ║
║  │                                                                         │ ║
║  │  # Merge on video_id                                                    │ ║
║  │  fused_df = cnn_df.merge(rppg_df, on='video_id')                       │ ║
║  │                                                                         │ ║
║  │  # Simple average fusion                                                │ ║
║  │  fused_df['P_final'] = (fused_df['P_CNN'] + fused_df['P_rPPG']) / 2    │ ║
║  │                                                                         │ ║
║  │  # Weighted fusion (if CNN is more accurate)                           │ ║
║  │  w_cnn, w_rppg = 0.6, 0.4                                              │ ║
║  │  fused_df['P_final'] = w_cnn * fused_df['P_CNN']                       │ ║
║  │                      + w_rppg * fused_df['P_rPPG']                      │ ║
║  │                                                                         │ ║
║  │  # Learned fusion (train a small classifier)                           │ ║
║  │  from sklearn.linear_model import LogisticRegression                   │ ║
║  │  X_fusion = fused_df[['P_CNN', 'P_rPPG']].values                       │ ║
║  │  y_fusion = fused_df['label'].values                                   │ ║
║  │  fusion_model = LogisticRegression().fit(X_fusion, y_fusion)           │ ║
║  │  fused_df['P_final'] = fusion_model.predict_proba(X_fusion)[:, 1]      │ ║
║  └─────────────────────────────────────────────────────────────────────────┘ ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("CNN SPATIO-TEMPORAL STREAM — FINAL SUMMARY")
print("="*70)

print(f"""
  ┌─────────────────────────────────────────────────────────────────────┐
  │  ARCHITECTURE                                                        │
  ├─────────────────────────────────────────────────────────────────────┤
  │  Backbone:           EfficientNet-B4 (pretrained)                   │
  │  Temporal Model:     BiLSTM + Multi-Head Self-Attention             │
  │  LSTM Hidden:        {cfg.LSTM_HIDDEN} × 2 (bidirectional)                           │
  │  LSTM Layers:        {cfg.LSTM_LAYERS}                                                │
  │  Attention Heads:    {cfg.ATTENTION_HEADS}                                                │
  │  Total Parameters:   {total_params:,}                                  │
  │  Trainable Params:   {trainable_params:,}                                  │
  └─────────────────────────────────────────────────────────────────────┘

  ┌─────────────────────────────────────────────────────────────────────┐
  │  TRAINING                                                           │
  ├─────────────────────────────────────────────────────────────────────┤
  │  Training Videos:    {len(train_dataset)}                                              │
  │  Validation Videos:  {len(val_dataset)}                                              │
  │  Frames per Video:   {cfg.FRAMES_PER_VIDEO}                                               │
  │  Best Epoch:         {best_epoch}                                                │
  │  Best Val AUC:       {best_val_auc:.4f}                                           │
  └─────────────────────────────────────────────────────────────────────┘

  ┌─────────────────────────────────────────────────────────────────────┐
  │  VIDEO-LEVEL METRICS                                                │
  ├─────────────────────────────────────────────────────────────────────┤
  │  AUC-ROC:            {video_auc:.4f}                                           │
  │  Accuracy:           {video_acc:.4f}                                           │
  │  F1-Score:           {video_f1:.4f}                                           │
  └─────────────────────────────────────────────────────────────────────┘

  ┌─────────────────────────────────────────────────────────────────────┐
  │  OUTPUT FILES                                                       │
  ├─────────────────────────────────────────────────────────────────────┤
  │  ✓ cnn_predictions.csv         (video-level P_CNN scores)          │
  │  ✓ best_cnn_model.pth          (best model weights)                │
  │  ✓ gradcam_gallery.png         (model interpretability)            │
  │  ✓ training_curves.png         (loss/acc/auc plots)                │
  └─────────────────────────────────────────────────────────────────────┘
""")

print("="*70)
print("\n✅ SPATIO-TEMPORAL CNN TRAINING COMPLETE!")
print("   ✓ Temporal modeling via BiLSTM + Attention (detects inter-frame artifacts)")
print("   ✓ Grad-CAM visualization (proves model learns meaningful features)")
print("   ✓ Ready for Late Fusion with rPPG physiological stream")
print("="*70)
